# 综合实训：电商舆情监控与负面评价预警系统

## 1. 学习目标

完成本课程后，你应该能够：

- 独立完成一个完整的手机评论分析项目
- 解决“评论太短”、“评分不平衡”等实际数据问题
- 训练一个能识别好评/差评的 AI 模型
- **核心技能**：自动提取差评中的“硬件故障点”（如提取出“屏幕”、“电池”等关键词）
- 制作词云图进行可视化展示

## 2. 项目介绍

### 2.1 项目背景

**场景设定：**  
你入职了一家数码电商公司。老板最近很头疼，新上架的一款手机虽然销量不错，但退货率在上升。老板把 100 多条用户评论扔给你，说：

“小王，你帮我分析一下：
1. 还有多少人在骂我们？（情感分析）
2. 他们到底在骂什么？是屏幕不行还是电池不行？（关键词提取）
3. 给我做张图，我要贴在产品经理脑门上。（可视化）”

### 2.2 数据集介绍

**数据文件：** `phone_reviews.csv`   


**数据集地址**：```/home/jovyan/work/datasets/691fcdcc6639511b893dc45f-momodel/phone_reviews.csv```

**数据字段：**
| 列名 | 说明 | 示例 |
|------|------|------|
| **rating** | 用户评分（1-5分） | 5, 1, 2 |
| **content** | 评论文本内容 | "刚买回来就发热严重，玩游戏简直是暖手宝..." |

## 3. 步骤1：加载和探索数据

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 1. 读取数据
df = pd.read_csv('/home/jovyan/work/datasets/691fcdcc6639511b893dc45f-momodel/phone_reviews.csv')

print("数据加载完成！")
print(f"\n数据集基本信息：")
print(f"总记录数: {len(df)} 条")

# 2. 查看前5条，找找感觉
print("\n前5条评论预览：")
print(df.head())

### 3.1 评分分布分析
我们先看看好评多还是差评多？这决定了我们后面要不要做数据平衡。

In [ ]:
# 统计各分数的数量
score_dist = df['rating'].value_counts().sort_index()

print("评分分布统计：")
print("="*40)
for score, count in score_dist.items():
    percentage = count / len(df) * 100
    bar = "█" * int(percentage / 2)
    print(f"{score}分: {count:<3}条 ({percentage:.1f}%) {bar}")

## 4. 步骤2：数据预处理

为了让 AI 学习更准确，我们需要：
1.  **打标签**：把 1-5 分转换成 0（差评）和 1（好评）。
2.  **分词**：把句子切成词语。

In [ ]:
import jieba

# 1. 定义转换规则：4-5分是好评(1)，1-2分是差评(0)，3分中立不管它
def make_label(rating):
    if rating >= 4: return 1
    if rating <= 2: return 0
    return -1

df['label'] = df['rating'].apply(make_label)

# 过滤掉3分的中立数据
df = df[df['label'] != -1]

print(f"清洗后剩余数据: {len(df)} 条")

# 2. 中文分词
print("正在进行分词...")
# 简单处理：用jieba切分，并用空格连接
df['clean_content'] = df['content'].apply(lambda x: " ".join(jieba.lcut(x)))

print(f"分词示例: {df['clean_content'].iloc[1]}")

## 5. 步骤3：模型训练

我们将使用 **朴素贝叶斯 (Naive Bayes)** 算法。它非常适合处理这种“看词猜意”的任务（比如看到“卡顿”就猜差评）。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_content'], df['label'], test_size=0.2, random_state=42
)

# 2. 特征提取 (把文字变数字)
vectorizer = TfidfVectorizer(max_features=1000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 3. 训练模型
print("正在训练模型...")
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# 4. 看看准确率
acc = accuracy_score(y_test, model.predict(X_test_vec))
print(f"\n✅ 模型训练完成！准确率: {acc:.2%}")

## 6. 步骤4：核心任务——关键词提取

这是老板最关心的部分：**用户到底在骂什么？**

我们将从差评中提取 **名词 (Noun)**。因为硬件故障通常都是名词（如“屏幕”、“电池”、“信号”）。

In [ ]:
import jieba.posseg as pseg
from collections import Counter

# 1. 筛选出所有差评
bad_reviews = df[df['label'] == 0]['content']

# 2. 提取名词 (n)
bad_nouns = []
stop_words = ['手机', '时候', '东西', '感觉'] # 这些词没意义，排除掉

for text in bad_reviews:
    words = pseg.cut(text)
    for word, flag in words:
        # flag.startswith('n') 表示是名词
        if flag.startswith('n') and len(word) > 1 and word not in stop_words:
            bad_nouns.append(word)

# 3. 统计前10大槽点
top_issues = Counter(bad_nouns).most_common(10)

print("📊 差评原因排行榜 (老板请看这里)：")
print("="*40)
for i, (word, count) in enumerate(top_issues, 1):
    bar = "▓" * count
    print(f"No.{i} {word:<5} {count}次 {bar}")

## 7. 步骤5：成果可视化（词云图）

最后，我们把刚才提取的关键词画成一张图，直观地展示给老板看。

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os

text_cloud = " ".join(bad_nouns)

# 1. 定义一个在 Linux 环境中常用的中文字体路径
# 大多数 Jupyter/Linux 环境都有这个字体
linux_font = '/usr/share/fonts/truetype/droid/DroidSansFallbackFull.ttf'

# 2. 检查字体是否存在，如果不存在则尝试使用当前目录下的字体，或者报错提示
if os.path.exists(linux_font):
    font_path = linux_font
elif os.path.exists('SimHei.ttf'): # 如果你自己上传了黑体文件
    font_path = 'SimHei.ttf'
else:
    # 如果都找不到，暂时不指定路径（可能会显示乱码方框，但不会报错）
    # 这里的逻辑是：先让程序跑通，再解决乱码
    font_path = None 
    print("⚠️ 警告：未找到中文字体文件，词云图中的中文可能会显示为方框（豆腐块）。")
    
# 配置词云
wc = WordCloud(
    font_path=font_path, # 使用上面检测到的路径
    background_color='white',
    width=800, height=400,
    colormap='Reds' # 差评用红色，警示！
).generate(text_cloud)

plt.figure(figsize=(10, 5))
plt.imshow(wc)
plt.axis('off')
plt.title("Mobile Phone Bad Review Analysis", fontsize=15)
plt.show()

## 8. 步骤6：应用模型预测新评论

让我们写一个简单的函数，模拟真实的“差评检测系统”。

In [ ]:
def review_crusher(text):
    # 1. 预处理
    vec = vectorizer.transform([" ".join(jieba.lcut(text))])
    # 2. 预测
    pred = model.predict(vec)[0]
    
    if pred == 1:
        return "✅ 好评"
    else:
        return "🚨 差评 (注意警惕!)"

# 模拟几条新数据
test_reviews = [
    "这手机太棒了，打游戏一点都不卡。",
    "什么垃圾手机，刚买就发热，退钱！",
    "屏幕有点模糊，不值这个价。",
    "物流很快，服务态度好。"
]

print("=== 差评检测系统运行中 ===")
for review in test_reviews:
    print(f"评论：{review}\n判定：{review_crusher(review)}\n------------------")

## 9. 项目总结

### 9.1 我们完成了什么？

1.  **数据清洗**：处理了手机评论数据，去除了中立评论。
2.  **情感分类**：训练了 AI 模型，能够自动区分好评和差评。
3.  **情报挖掘**：通过**词性标注 (POS Tagging)**，精准提取了差评中的硬件故障点（屏幕、电池等）。
4.  **可视化报告**：生成了让老板一目了然的词云图。

### 9.2 商业价值

通过这个系统，产品经理不需要看几千条评论，只需要看一眼词云图，就知道下一代手机需要改进**信号**和**屏幕**。这就是数据分析的价值！